In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from pipeline_helpers import ARTIFACT_DIR, DATA_PATH, SEED

print(f"Data: {DATA_PATH}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Reproducibility seed: {SEED}")
from pipeline_helpers import phase4
result = phase4()
display(result["choice_table"])

Data: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\data\raw\europe_cereal_yield_climate_1990_2022.csv
Artifacts: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline
Reproducibility seed: 42


,choice,decision
0,Split,"One fixed 80/20 random holdout, stratified by ITEM, random_state=42; rows are shuffled observations, not chronological blocks."
1,Validation,5-fold shuffled KFold CV on training rows for tuning; held-out test set is used once for final comparison.
2,Scaling,StandardScaler for Ridge numeric columns; tree models use numeric values without scaling. Country/crop are one-hot encoded in both contexts for comparable preprocessing.
3,Context ablation,Fit each candidate both with and without AREA and ITEM one-hot features.


## Fixed split and category balance

A single 80/20 random holdout is stratified by crop (`random_state=42`). Five-fold shuffled K-fold CV on training rows is reserved for tuning; no chronological splitting, lagging, or rolling features are used.

In [2]:
display(result["counts"])
display(result["stratification"])
display(result["numeric_summary"])
display(result["feature_list"])

,partition,rows
0,train,2596
1,test,649


col_0,ITEM,test,train
0,Barley,0.362096,0.362866
1,Maize (corn),0.277350,0.276965
2,Wheat,0.360555,0.360169


,feature,count,mean,std,min,25%,50%,75%,max
0,WB_CCKP_CDD,3245.0,23.867217,13.616640,10.15,16.81,20.22,25.69,163.99
1,WB_CCKP_CSDI,3245.0,6.603322,7.408598,0.00,1.32,4.82,9.24,57.40
2,WB_CCKP_CWD,3245.0,10.253812,2.597230,4.00,8.41,9.67,11.50,22.26
3,WB_CCKP_HURS,3245.0,75.231831,4.401734,60.68,72.21,75.64,78.58,84.84
4,WB_CCKP_PR,3245.0,898.732561,288.095694,283.40,703.06,831.12,1029.12,2274.21
5,WB_CCKP_R50MM,3245.0,0.338980,0.630002,0.00,0.01,0.07,0.39,5.91
6,WB_CCKP_RX5DAY,3245.0,73.601390,24.676011,34.59,54.89,67.05,88.46,192.31
7,WB_CCKP_TNN,3245.0,-13.758068,8.520754,-43.30,-18.61,-13.95,-7.93,11.64
8,WB_CCKP_TR,3245.0,45.832931,34.795731,0.00,17.23,37.76,72.28,177.32
9,WB_CCKP_TR29,3245.0,0.691245,1.549065,0.00,0.00,0.01,0.48,13.30


,selected_numeric_feature
0,WB_CCKP_CDD
1,WB_CCKP_CSDI
2,WB_CCKP_CWD
3,WB_CCKP_HURS
4,WB_CCKP_PR
5,WB_CCKP_R50MM
6,WB_CCKP_RX5DAY
7,WB_CCKP_TNN
8,WB_CCKP_TR
9,WB_CCKP_TR29


The split indices are saved and reused by each candidate and feature-set comparison. Ridge and the Phase 8 XGBoost candidate scale numeric columns inside their pipelines; baseline tree models use unscaled numeric inputs. Each scaler is fitted within the training fold. AREA and ITEM are compared both included and excluded.